In [ ]:
# !pip -q install tqdm

In [ ]:
# @title Full Dataset and Dataloader

!pip install tqdm
from datasets import load_dataset
import pandas as pd
import torch
from torch.utils.data import Dataset
from huggingface_hub import hf_hub_download
from ast import literal_eval
from tqdm import tqdm
import numpy as np
tqdm.pandas()

class HiddenStatesDataset(Dataset):
    def __init__(self, X_train, y_train):
        self.df = X_train
        self.labels = y_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        hidden_states = self.df.iloc[idx]
        accuracy = self.labels.iloc[idx]
        return torch.tensor(hidden_states, dtype=torch.float32), torch.tensor(accuracy, dtype=torch.float32)

def load_train_split(filename: str) -> pd.DataFrame:
    path = hf_hub_download(
        repo_id="dhgottesman/keen_estimating_knowledge_in_llms",
        filename=filename,
        repo_type="dataset",
    )
    return pd.read_csv(path, index_col=0)

def split_dataset_into_train_val_test(dataset, features="hidden_states"):
    train_subjects = load_train_split("popqa_train_subjects.csv")
    val_subjects = load_train_split("popqa_val_subjects.csv")
    test_subjects = load_train_split("popqa_test_subjects.csv")

    train_df = dataset.merge(train_subjects, on="subject").dropna()
    val_df = dataset.merge(val_subjects, on="subject").dropna()
    test_df = dataset.merge(test_subjects, on="subject").dropna()

    X_train = train_df[features]
    y_train = train_df["accuracy"]
    X_val = val_df[features]
    y_val = val_df["accuracy"]
    X_test = test_df[features]
    y_test = test_df["accuracy"]
    return X_train, y_train, X_val, y_val, X_test, y_test


repo_id = "kokolamba/keen_popqa_gpt2xl_generations"
path_in_repo = "features/residual_stream.parquet"

local_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=path_in_repo,
)

qa_dataset = pd.read_parquet(local_path)
qa_dataset.head()

features/residual_stream.parquet:   0%|          | 0.00/128M [00:00<?, ?B/s]

,subject,s_uri,label,total_examples,accuracy,layer_0,layer_1,layer_2,hidden_states
0,'71,http://www.wikidata.org/entity/Q12100227,head,7,0.142857,"[0.6265129446983337, 0.6194421648979187, 0.428...","[0.6288121342658997, 0.5981050133705139, 0.418...","[0.6383209824562073, 0.6060779690742493, 0.425...","[0.6312153339385986, 0.6078750491142273, 0.423..."
1,(I Can't Get No) Satisfaction,http://www.wikidata.org/entity/Q158553,head,5,0.000000,"[0.4077925980091095, 0.6454039812088013, 0.320...","[0.3738816976547241, 0.6309443116188049, 0.331...","[0.38318538665771484, 0.6324501037597656, 0.32...","[0.3882865607738495, 0.6362661719322205, 0.327..."
2,10,http://www.wikidata.org/entity/Q184591,head,7,0.000000,"[0.4725494980812073, 0.4378793239593506, 0.634...","[0.5120527744293213, 0.4383334517478943, 0.607...","[0.5325338244438171, 0.4527283012866974, 0.631...","[0.5057120323181152, 0.4429803788661957, 0.624..."
3,10 Years,http://www.wikidata.org/entity/Q2579741,head,7,0.285714,"[0.7377564311027527, 0.6917906403541565, 0.255...","[0.758429229259491, 0.6873093843460083, 0.2818...","[0.7618278861045837, 0.7357169389724731, 0.340...","[0.7526711821556091, 0.704939067363739, 0.2926..."
4,13,http://www.wikidata.org/entity/Q3018412,head,5,0.200000,"[0.5211764574050903, 0.45791491866111755, 0.54...","[0.5151306986808777, 0.44285085797309875, 0.50...","[0.5261474847793579, 0.4651433825492859, 0.533...","[0.5208181738853455, 0.45530304312705994, 0.52..."


In [ ]:
len(qa_dataset['hidden_states'][0])

1600

# Linear Probe

We define the probe as one linear layer: `f(z, θ) := θ · z`.

Where:

- `z` is the hidden representation vector for an entity
- `θ` is a learnable parameter vector (same size as `z`)

In training, we aim to increase the correlation between the score predicted by the probe and the gold **label** metric (QA accuracy).


In [ ]:
# @title LinearProbe
import torch.nn as nn
import torch.optim as optim
import copy
from scipy.stats import pearsonr
import wandb
from tqdm import tqdm

class LinearProbe(nn.Module):
    def __init__(self, input_size, learning_rate, max_iter, weight_decay=0.01):
        super().__init__()
        self.max_iter = max_iter
        self.criterion = nn.MSELoss()
        self.layer = nn.Linear(input_size, 1, bias=False)
        self._initialize_weights()

        self.optimizer = optim.AdamW(self.parameters(), lr=learning_rate, weight_decay=weight_decay)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=self.max_iter)
        self.best_weights = copy.deepcopy(self.layer.state_dict())
        self.best_val_corr = -1
        self.final_val = None

        self.cuda()

    def _initialize_weights(self):
        nn.init.kaiming_uniform_(self.layer.weight, nonlinearity="relu")

    def forward(self, x):
        return torch.sigmoid(self.layer(x))

    def predict(self, x):
        with torch.no_grad():
            return self.forward(x)

    def validate(self, X_val, y_val):
        preds = self.predict(X_val)
        y_val = y_val.reshape(-1, 1)
        loss = self.criterion(preds, y_val).item()
        preds_np = preds.squeeze(-1).cpu().numpy()
        targets_np = y_val.squeeze(-1).cpu().numpy()
        corr, p_val = pearsonr(preds_np, targets_np)

        result_df = pd.DataFrame({"preds": preds_np, "target": targets_np})
        return result_df, loss, corr, p_val

    def set_to_best_weights(self):
        self.layer.load_state_dict(self.best_weights)

    def fit(self, X_train, y_train, X_val, y_val):
        X_val = torch.tensor(X_val, dtype=torch.float32).cuda()
        y_val = torch.tensor(y_val, dtype=torch.float32).cuda()

        for epoch in tqdm(range(self.max_iter), total=self.max_iter):
            self.train()
            total_loss = 0

            for batch_x, batch_y in X_train:
                self.optimizer.zero_grad()
                batch_x = batch_x.to(torch.float32).cuda()
                batch_y = batch_y.to(torch.float32).reshape(-1, 1).cuda()

                preds = self.forward(batch_x)
                loss = self.criterion(preds, batch_y)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()

            if self.scheduler:
                self.scheduler.step()

            avg_train_loss = total_loss / len(X_train)
            result_df, val_loss, corr, _ = self.validate(X_val, y_val)
            if corr > self.best_val_corr:
                self.best_val_corr = corr
                self.best_weights = copy.deepcopy(self.layer.state_dict())
                self.final_val = result_df

            wandb.log({"train_loss": avg_train_loss, "val_loss": val_loss, "val_pearson_corr": corr})

        wandb.finish()

In [ ]:
# @title Training Configuration
from torch.utils.data import DataLoader

input_size = 1600
learning_rate = 0.00001 # @param {type:"raw"}
max_iter = 500 # @param {type:"integer"}
batch_size = 16 # @param {type:"integer"}
weight_decay = 0.01 # @param {type:"raw"}

dataset = qa_dataset
X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_into_train_val_test(dataset)

train_dataset = HiddenStatesDataset(X_train, y_train)
dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=1)

In [ ]:
# @title Connecting/Initializing wandb
!wandb login # PASTE WANDB API KEY

wandb.init(
    project="koko-gp2-xl-linear-probe",   # REQUIRED → all runs grouped under this project
    name="run-12",                 # optional: gives this run a readable name
    config={                      # optional: hyperparameters & settings
        "learning_rate": learning_rate,
        "max_iter": max_iter,
        "batch_size": batch_size,
        "weight_decay": weight_decay, # Added weight_decay to wandb config
        "model": "LinearProbe"
    }
)

wandb: Currently logged in as: abdulhakeemadefioye (abdulhakeemadefioye-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


- run-1 has lr=1e-5, max_iter=5 and batch_size=32 with sigmoid and cosine scheduling with no weight decay. corr = -0.15
- run-2 has lr=1e-5, max_iter=5 and batch_size=32 with sigmoid and cosine scheduling with wd=0.01. corr = 0.02
- run-3 has lr=1e-5, max_iter=5 and batch_size=1 with sigmoid and cosine scheduling with wd=0.01. corr = 0.23
- run-4 has lr=1e-5, max_iter=5 and batch_size=4 with sigmoid and cosine scheduling with wd=0.01. corr = -0.15
- run-5 has lr=1e-5, max_iter=5 and batch_size=2 with sigmoid and cosine scheduling with wd=0.01. corr = 0.08
- run-6 has lr=1e-5, max_iter=100 and batch_size=1 with sigmoid and cosine scheduling with wd=0.01. corr = 0.62
- run-7 has lr=1e-5, max_iter=500 and batch_size=1 with sigmoid and cosine scheduling with wd=0.01. corr = 0.64
- run-8 has lr=1e-5, max_iter=500 and batch_size=2 with sigmoid and cosine scheduling with wd=0.01. corr = 0.64(No shuffling in dataloader so far)
- run-9 has lr=1e-5, max_iter=500 and batch_size=2 with sigmoid and cosine scheduling with wd=0.01. corr = 0.64(With shuffling in dataloader)
- run-10 has lr=1e-5, max_iter=500 and batch_size=32 with sigmoid and cosine scheduling with wd=0.01. corr = 0.62 (With shuffling in dataloader)
- run-11 has lr=1e-5, max_iter=500 and batch_size=1 with sigmoid and cosine scheduling with wd=0.01. corr = 0.64  (With shuffling in dataloader)
- run-12 has lr=1e-5, max_iter=500 and batch_size=16 with sigmoid and cosine scheduling with wd=0.01. corr = 0.63 (With shuffling in dataloader)

### TODO:
- Rerun the run-7 with batch_size of 32 to see if the corr improves significantly
- Investigate why batch_size > 1 seems to be performing very poorly at relatively small number of epochs contrary to that of batch_size = 1.

>NOTE: correlation is evaluated on test set.

In [ ]:
# @title Kicking Off Training

probe = LinearProbe(input_size, learning_rate, max_iter, weight_decay=weight_decay)
probe.fit(dataloader, y_train, X_val, y_val)

100%|██████████| 500/500 [04:30<00:00,  1.85it/s]


train_loss,█▆▆▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▅▅▅▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_pearson_corr,▁▂▃▄▆▆▆▇▇▇▇▇████████████████████████████
train_loss,0.03798
val_loss,0.0465
val_pearson_corr,0.62846


In [ ]:
probe.set_to_best_weights()

In [ ]:
# @title Compute the Pearson Correlation on the test set.

_X_test = torch.tensor(X_test, dtype=torch.float32).cuda()
_y_test = torch.tensor(y_test, dtype=torch.float32).cuda()

result_df, _, corr, _ = probe.validate(_X_test, _y_test)
print(f"Test correlation {corr:.2f}")

Test correlation 0.63


In [ ]:
torch.optim.lr_scheduler.CosineAnnealingLR??